# Train/test split and scaling

one stratified 80/20 split shared by every model and feature set and a scaler fitted (on the training rows only)

In [ ]:
import sys
sys.path.append('..')

import numpy as np
import pandas as pd
from src.config import DATA_PATH
from src.imbalance import create_imbalance
from src.features import encode_features, select_feature_sets
from src.split import split_indices, scale

target = 'Performance_Score'
df = pd.read_csv(DATA_PATH)
df_imb = create_imbalance(df, target)

X = encode_features(df_imb, target)
y = df_imb[target]
sets = select_feature_sets(X)

## Stratified split

`stratify=y` keeps the same class proportions in train and test. Without it, class 1 (0.6 % of the rows) could end up almost absent from one side. The split is done once, on row positions, so that the `salary` and `no_salary` sets are evaluated on exactly the same rows.

In [ ]:
idx_train, idx_test = split_indices(y)
y_train, y_test = y.iloc[idx_train], y.iloc[idx_test]

print('train :', len(idx_train), ' test :', len(idx_test))
assert len(idx_train) + len(idx_test) == len(y)
assert len(idx_test) == int(np.ceil(0.2 * len(y)))
assert not set(idx_train) & set(idx_test)

In [ ]:
summary = pd.DataFrame({
    'train': y_train.value_counts().sort_index(),
    'test': y_test.value_counts().sort_index(),
})
summary['train_share'] = (summary['train'] / summary['train'].sum()).round(4)
summary['test_share'] = (summary['test'] / summary['test'].sum()).round(4)
summary

In [ ]:
assert (summary['train_share'] - summary['test_share']).abs().max() < 0.001